In [10]:
import sys
!{sys.executable} -m pip install groq -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [12]:
from groq import Groq

client = Groq(api_key="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")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # free, fast, very capable
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm looking forward to assisting you with any questions or topics you'd like to discuss.


In [16]:
import json
import re
from groq import Groq

client = Groq(api_key="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")

# Mock patient row — we'll replace this with real X_test data in Step 3
mock_patient = {
    "age": 67,
    "heart_rate": 112,
    "systolic_bp": 88,
    "temperature": 38.9,
    "chief_complaint": "chest pain and shortness of breath",
    "arrival_mode": "ambulance",
    "comorbidities": "diabetes, hypertension"
}

mock_risk_score = 0.87  # pretend your ML model returned this

prompt = f"""
You are an emergency department triage assistant.

Patient data: {mock_patient}
ML model risk score (0 to 1, higher = more critical): {mock_risk_score}

Respond ONLY with valid JSON in this exact format:
{{
  "urgency_level": "emergent",
  "recommended_actions": ["action 1", "action 2", "action 3"],
  "triage_note": "1-2 sentence explanation of urgency and key risk factors."
}}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2  # low temp = more consistent, structured output
)

raw = response.choices[0].message.content.strip()

# Strip markdown code fences if model wraps response in ```json ... ```
if raw.startswith("```"):
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("```").strip()

decision = json.loads(raw)

print("Urgency     :", decision["urgency_level"])
print("Actions     :", decision["recommended_actions"])
print("Triage Note :", decision["triage_note"])

Urgency     : emergent
Actions     : ['immediate ECG', 'administer oxygen', 'notify cardiologist for potential STEMI']
Triage Note : The patient's high ML model risk score, chest pain, shortness of breath, and abnormal vital signs indicate a high-risk condition requiring immediate attention. The patient's comorbidities, including diabetes and hypertension, further increase the urgency of the situation.


In [38]:
import joblib

rf_model = joblib.load(r'C:\Users\ADMIN\Documents\ML Self learn\ML\Hospital data\3 - DeploymentAPI\risk_model.pkl')
print("Model loaded:", rf_model)

Model loaded: RandomForestClassifier(class_weight='balanced', max_depth=10, random_state=42)


In [42]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

df = pd.read_csv(r'C:\Users\ADMIN\Documents\ML Self learn\ML\Hospital data\1 - Analytics and EDA\model_table.csv')
df['visit_date'] = pd.to_datetime(df['visit_date'])

le = LabelEncoder()
y = pd.Series(le.fit_transform(df['risk_score']))

features = [
    'age', 'length_of_stay_hours', 'visit_frequency', 'avg_los_per_patient',
    'days_since_registration', 'visit_month', 'visit_dayofweek', 'chronic_flag',
    'department_enc', 'visit_type_enc', 'gender_enc', 'city_enc'
]

X = df[['visit_date'] + features].copy()
X.fillna(X.median(numeric_only=True), inplace=True)
X = X.sort_values('visit_date')
y = y.loc[X.index]

split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx].drop(columns=['visit_date'])
X_test  = X.iloc[split_idx:].drop(columns=['visit_date'])

print("X_test shape:", X_test.shape)


X_test shape: (5000, 12)


In [48]:
import json
import re
from groq import Groq

client = Groq(api_key="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")

# --- Your actual variables ---
model_ml = rf_model   # Risk classification model
feature_names = list(X_train.columns)
X_sample = X_test.iloc[0].values
# -----------------------------

# rf_model predicts 3 classes: 0=High, 1=Low, 2=Medium
# We want probability of "High" risk → class index 0
proba = model_ml.predict_proba([X_sample])[0]
risk_score = float(proba[0])  # probability of High risk
predicted_class = {0: "High", 1: "Low", 2: "Medium"}[model_ml.predict([X_sample])[0]]

patient_dict = dict(zip(feature_names, X_sample))

prompt = f"""
You are an emergency department triage assistant.

Patient data: {patient_dict}
Predicted risk class: {predicted_class}
Probability of High risk: {risk_score:.3f}

Respond ONLY with valid JSON in this exact format:
{{
  "urgency_level": "emergent",
  "recommended_actions": ["action 1", "action 2", "action 3"],
  "triage_note": "1-2 sentence explanation of urgency and key risk factors."
}}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

raw = response.choices[0].message.content.strip()
if raw.startswith("```"):
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("```").strip()

decision = json.loads(raw)

print(f"Predicted Class : {predicted_class}")
print(f"High Risk Prob  : {risk_score:.3f}")
print(f"Urgency         : {decision['urgency_level']}")
print(f"Actions         : {decision['recommended_actions']}")
print(f"Triage Note     : {decision['triage_note']}")

C:\Users\ADMIN\anaconda3\envs\tf_gpu_env\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ADMIN\anaconda3\envs\tf_gpu_env\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Predicted Class : Low
High Risk Prob  : 0.263
Urgency         : non-emergent
Actions         : ['routine evaluation', 'monitor patient condition', 'schedule follow-up appointment']
Triage Note     : The patient is classified as low risk with a probability of high risk being 0.263, and can be treated on a non-emergent basis, but should still be monitored for any changes in condition.


In [67]:
%run pre_compute.py

✓ Patient P1000 done — Low / non_urgent
✓ Patient P1001 done — Medium / urgent
✓ Patient P1002 done — Medium / urgent
✓ Patient P1003 done — High / urgent
✓ Patient P1004 done — Medium / urgent
✓ Patient P1005 done — Low / non_urgent
✓ Patient P1006 done — Medium / urgent
✓ Patient P1007 done — Low / non_urgent
✓ Patient P1008 done — High / urgent
✓ Patient P1009 done — Low / non_urgent
✓ Patient P1010 done — Low / non_urgent
✓ Patient P1011 done — Low / non_urgent
✓ Patient P1012 done — High / urgent
✓ Patient P1013 done — Medium / urgent
✓ Patient P1014 done — Low / non_urgent
✓ Patient P1015 done — Low / non_urgent
✓ Patient P1016 done — Medium / urgent
✓ Patient P1017 done — Low / non_urgent
✓ Patient P1018 done — Low / non_urgent
✓ Patient P1019 done — Low / non_urgent

✅ triage_decisions.json saved with 20 patients.


In [69]:
import sys
!{sys.executable} -m pip install streamlit -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
